In [4]:
import pandas as pd
import re
import json
import csv
import os
from thefuzz import fuzz
import unicodedata
import glob
from tqdm import tqdm
from datetime import datetime
from rapidfuzz import fuzz
import sys

## Sofifa data

In [3]:
def clean_player_data(input_csv,output_csv):
    df = pd.read_csv(input_csv)

    # Helper function to safely extract regex groups
    def extract_field(pattern, text):
        match = re.search(pattern, str(text))
        return match.group(1).strip() if match else None

    # Extracting core Stats
    df['Age'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'(\d+)y\.o\.', x))
    df['DOB'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'\(([A-Z][a-z]{2}\s\d{1,2},\s\d{4})\)', x))
    df['Height_cm'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'(\d+)cm', x))
    df['Weight_kg'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'(\d+)kg', x))
    df['Overall_Rating'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'(\d+)\s*\|\s*Overall rating', x))
    df['Potential_Rating'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'(\d+)(?:\s*[+-]\d+)?\s*\|\s*Potential', x))
    df['Value_EUR'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'(€[\d\.KMB]+)\s*\|\s*Value', x))
    df['Wage_EUR'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'(€[\d\.KMB]+)\s*\|\s*Wage', x))
    df['Preferred_Foot'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'Preferred foot\s([A-Za-z]+)', x))

   #ATTRIBUTE EXTRACTION
    #| number Attribute Name |
    attr_pattern = r'\|\s*(\d{1,3})\s+([A-Za-z\s]+?)\s*\|'
    
    all_attributes_list = []
    
    for text in df['Raw_Card_Text']:
        matches = re.findall(attr_pattern, str(text))

        # Building a dictionary for the row, cleaning up spaces in column names
        row_attrs = {attr_name.strip().replace(' ', '_'): int(val) for val, attr_name in matches}
        all_attributes_list.append(row_attrs)
        
    attrs_df = pd.DataFrame(all_attributes_list)     # Converting the list of dictionaries into a new DataFrame

    # Combination and Clean Up
    # Concatenate the dynamic attribute columns to the main DataFrame
    df = pd.concat([df, attrs_df], axis=1)
    
    # Converting numeric columns to actual numeric types for the database
    numeric_cols = ['Age', 'Height_cm', 'Weight_kg', 'Overall_Rating', 'Potential_Rating']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Dropping raw string column
    df_cleaned = df.drop(columns=['Raw_Card_Text'])
    df_cleaned.to_csv(output_csv,index=False)
    display(df_cleaned)
    return df_cleaned



In [4]:

cleaned_df=clean_player_data("data/sofifa/sofifa_raw_data_COMPLETE.csv","data/sofifa/sofifa_cleaned.csv")

,URL,Name,Age,DOB,Height_cm,Weight_kg,Overall_Rating,Potential_Rating,Value_EUR,Wage_EUR,...,Sliding_tackle,GK_Diving,GK_Handling,GK_Kicking,GK_Positioning,GK_Reflexes,Marking,Positioning,Tactical_awareness,Tackling
0,https://sofifa.com/player/272465/caleb-roberts...,Caleb Roberts,19,"Oct 24, 2005",178,74,56,70,€350K,€600,...,42.0,14.0,6.0,13.0,13.0,7.0,NaN,NaN,NaN,NaN
1,https://sofifa.com/player/random?1775801033,Javier Villar del Fraile,22,"Mar 15, 2003",187,69,65,74,€1.6M,€2K,...,67.0,8.0,6.0,8.0,7.0,15.0,NaN,NaN,NaN,NaN
2,https://sofifa.com/player/274482/adrian-ascues...,Adrián Ascues,21,"Nov 15, 2002",181,81,65,73,€1.6M,€12K,...,41.0,8.0,14.0,6.0,11.0,8.0,NaN,NaN,NaN,NaN
3,https://sofifa.com/player/272535/josh-bailey/2...,Josh Bailey,18,"Jun 24, 2005",179,72,52,63,€160K,€500,...,44.0,9.0,11.0,11.0,8.0,10.0,NaN,NaN,NaN,NaN
4,https://sofifa.com/player/271880/nestor-jimene...,Nestor Jiménez,22,"Apr 8, 2003",175,66,61,70,€750K,€4K,...,18.0,10.0,13.0,12.0,6.0,8.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6994,https://sofifa.com/player/272834/joao-pedro-go...,João Pedro Gonçalves Neves,20,"Sep 27, 2004",174,66,87,92,€117.5M,€110K,...,82.0,8.0,7.0,5.0,13.0,13.0,NaN,NaN,NaN,NaN
6995,https://sofifa.com/player/276694/johan-manzamb...,Johan Manzambi,19,"Oct 14, 2005",183,75,75,85,€12M,€18K,...,67.0,8.0,10.0,8.0,13.0,9.0,NaN,NaN,NaN,NaN
6996,https://sofifa.com/player/263765/tom-bischof/2...,Tom Bischof,20,"Jun 28, 2005",180,75,78,86,€30.5M,€41K,...,66.0,9.0,13.0,12.0,13.0,6.0,NaN,NaN,NaN,NaN
6997,https://sofifa.com/player/234826/antonio-marti...,Antonio Martínez López,28,"Jun 30, 1997",187,83,1,76,€7.5M,€31K,...,23.0,11.0,11.0,12.0,7.0,9.0,NaN,NaN,NaN,NaN


## soccersolver data

### Player Data

In [2]:
json_folder = 'data/soccersolver-player-data/players' 

# The name of the final file you want to create
output_csv = 'data/soccersolver-player-data/unified/merged_players.csv'

headers = [
    "player_id", "name", "team", "team_id", "position", 
    "main_position", "age", "birth_date", "nationality", 
    "height", "preferred_foot", "shirt_number", "market_value", 
    "img_url", "profile_url", "season"
]

all_players_data = []

for filename in os.listdir(json_folder):
    if filename.endswith('.json'):
        print(f"Processing: {filename}")
        file_path = os.path.join(json_folder, filename)
        
        with open(file_path, 'r', encoding='utf-8') as file:
            data = json.load(file)
            
            if isinstance(data, dict):
                data = [data]            # This handles both cases: if the JSON is a single player dict or a list of multiple player dicts.
                
            for player in data:
                clean_player = {key: player.get(key, '') for key in headers}        # Create a clean dictionary for each player keeping only our defined headers If a field is missing in the JSON, it defaults to an empty string ('')
                all_players_data.append(clean_player)

with open(output_csv, 'w', newline='', encoding='utf-8') as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=headers)
    writer.writeheader()
    writer.writerows(all_players_data)

print(f"\Compiled {len(all_players_data)} player records into '{output_csv}'.")

<>:35: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
<>:35: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
C:\Users\ashwy\AppData\Local\Temp\ipykernel_13932\2360835386.py:35: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
  print(f"\Compiled {len(all_players_data)} player records into '{output_csv}'.")


Processing: players_all_2019-2020.json
Processing: players_all_2020-2021.json
Processing: players_all_2021-2022.json
Processing: players_all_2022-2023.json
Processing: players_all_2023-2024.json
Processing: players_all_2024-2025.json
Processing: players_all_2025-2026.json
\Compiled 405290 player records into 'data/soccersolver-player-data/unified/merged_players.csv'.


### League Data

In [ ]:
season_files_path = 'data/soccersolver-player-data/leagues/season_files/*.json'     # Folder containing your 7 season .json files
master_file_path = 'data/soccersolver-player-data/leagues/discovered_leagues.json'    # master file with general information

output_csv = 'data/soccersolver-player-data/unified/unified_leagues.csv'  # final output file

with open(master_file_path, 'r', encoding='utf-8') as f:
    master_data = json.load(f)

if isinstance(master_data, dict):
    # Scenario A: a dict of dicts 
    if all(isinstance(v, dict) for v in master_data.values()):
        records = list(master_data.values())
    # Scenario B: wrapped inside a key 
    else:
        records = []
        for key, value in master_data.items():
            if isinstance(value, list):
                records = value
                break
elif isinstance(master_data, list):
    # Scenario C:  already a clean list
    records = master_data
else:
    records = [master_data]

master_df = pd.DataFrame(records)

if 'league_id' not in master_df.columns:
    print("'league_id' not found in master data.")
    print("Columns extracted:", list(master_df.columns))
    print("Sample record:", records[0] if records else "Empty")
    raise KeyError("The master JSON structure is completely unrecognized. Check the print outputs above.")

# the specific columns we want to inject
desired_columns = ['league_id', 'url_path', 'value_m', 'region', 'is_youth']
existing_columns = [col for col in desired_columns if col in master_df.columns]

master_static_data = master_df[existing_columns]

# Ingesting and Compiling Season Fact Tables 
season_files = glob.glob(season_files_path)

# List comprehension to read all JSONs into DataFrames, then concatenate them into one huge table
season_dfs = [pd.read_json(file) for file in season_files]
all_seasons_df = pd.concat(season_dfs, ignore_index=True)

# Hash Join
# merges based on the matching 'league_id'.
final_df = pd.merge(all_seasons_df, master_static_data, on='league_id', how='left')
print(final_df.head())

  league_id                 name   country     season   tier  \
0      IT3A   Serie C - Girone A     Italy  2019-2020      3   
1      CLPD      Liga de Primera     Chile  2019-2020      1   
2      PT23   Liga Revelação U23  Portugal  2019-2020  youth   
3        L3              3. Liga   Germany  2019-2020      3   
4      MLS1  Major League Soccer       USA  2019-2020      1   

   total_market_value  num_teams  num_players  average_age  \
0           106600000         20          564         24.5   
1           158330000         16          425         27.0   
2            11475000         17          527         20.2   
3           155225000         20          589         25.4   
4          1319650000         30          845         25.4   

   average_market_value most_valuable_player  \
0                189000                        
1                373000                        
2                 22000                        
3                264000                        
4 

In [8]:
final_df.to_csv(output_csv, index=False, encoding='utf-8')

print(f"Success! Processed {len(all_seasons_df)} season records and saved to {output_csv}.")

Success! Processed 525 season records and saved to data/soccersolver-player-data/unified/unified_leagues.csv.


### Team Data

In [3]:
team_json_path = 'data/soccersolver-player-data/teams/*.json'   
output_csv = 'data/soccersolver-player-data/unified/unified_teams.csv'

all_team_records = []

files = glob.glob(team_json_path)

for file_path in files:
    print(f"Processing: {os.path.basename(file_path)}")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # unwrapping logic: handles lists of teams or dicts of teams
    if isinstance(data, dict):
        # If it's a dict of dicts, taking the values
        if all(isinstance(v, dict) for v in data.values()):     
            records = list(data.values())
        else:
            # Checking for common wrapper keys like 'teams' or 'data'
            records = data.get('teams', data.get('data', [data]))
    elif isinstance(data, list):
        records = data
    else:
        records = [data]
        
    all_team_records.extend(records)

teams_df = pd.json_normalize(all_team_records)    # handles any missing fields by inserting NaN automatically

columns_to_keep = [
    "team_id", "name", "league", "league_id", "country", "season",
    "squad_size", "average_age", "total_market_value", "average_market_value",
    "foreign_players_count", "national_players_count", "stadium_name",
    "stadium_capacity", "logo_url", "profile_url"
]
existing_cols = [c for c in columns_to_keep if c in teams_df.columns]   # Only keeping columns that actually exist in the data to avoid errors
final_df = teams_df[existing_cols]
print(final_df.head())

Processing: teams_all_2019-2020.json
Processing: teams_all_2020-2021.json
Processing: teams_all_2021-2022.json
Processing: teams_all_2022-2023.json
Processing: teams_all_2023-2024.json
Processing: teams_all_2024-2025.json
Processing: teams_all_2025-2026.json
  team_id                             name                league league_id  \
0   27190                    Shanghai Port  Chinese Super League       CSL   
1   10948       Guangzhou FC (1954 - 2024)  Chinese Super League       CSL   
2    3176                    Beijing Guoan  Chinese Super League       CSL   
3   33304  Dalian Professional (2009-2024)  Chinese Super League       CSL   
4    3182                 Shandong Taishan  Chinese Super League       CSL   

  country     season  squad_size  average_age  total_market_value  \
0   China  2019-2020          60         22.9         110350000.0   
1   China  2019-2020          55         22.2          92630000.0   
2   China  2019-2020          48         22.7          77230000.0

In [4]:
final_df.to_csv(output_csv, index=False, encoding='utf-8')
print(f"\nCombined {len(final_df)} team entries into '{output_csv}'.")


Combined 9926 team entries into 'data/soccersolver-player-data/unified/unified_teams.csv'.


## Wyscoiut Data

In [2]:
input_folder = 'data/wyscout_data/raw/'
output_folder = 'data/wyscout_data/cleaned/'

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Loading areas and renaming columns to avoid overlapping names during merges
areas_df = pd.read_csv(os.path.join(input_folder, 'areas.csv'))
areas_map = areas_df[['id', 'name', 'alpha3code']].copy()
areas_map.columns = ['geo_id', 'mapped_country', 'mapped_alpha3']


In [3]:
# Helper Function for JSON 
def extract_wyscout_id(json_string): # Safely extracts the integer ID from the messy JSON strings.
    try:
        if pd.isna(json_string):
            return None     # Convert single quotes to double quotes just in case, then parse
        clean_str = str(json_string).replace("'", '"')
        parsed = json.loads(clean_str)
        return parsed.get("wyscout")
    except Exception:
        return None

In [4]:
comps_df = pd.read_csv(os.path.join(input_folder, 'competitions.csv'))
comps_clean = pd.merge(comps_df, areas_map, left_on='area_id', right_on='geo_id', how='left')

comps_clean['country'] = comps_clean['mapped_country']
comps_clean['country_code'] = comps_clean['mapped_alpha3']
comps_clean['wyscout_id'] = comps_clean['external_ids'].apply(extract_wyscout_id)

# Purge technical and redundant columns
comps_drop = ['area_id', 'geo_id', 'mapped_country', 'mapped_alpha3', 'external_ids', 'created_at', 'updated_at']
comps_clean.drop(columns=[col for col in comps_drop if col in comps_clean.columns], inplace=True)
print(comps_clean.head())

   id                                        name                   format  \
0   1                       SAFF U19 Championship        International cup   
1   2  EAFF E-1 Football Championship Preliminary        International cup   
2   3                     ASEAN Club Championship        International cup   
3   4                      UAE-Qatar Super Shield  International super cup   
4   5                         Qatar-UAE Super Cup  International super cup   

  gender category           type  division_level  priority_level country  \
0   male      U19  international               0               1    Asia   
1   male  default  international               1               1    Asia   
2   male  default           club               1               1    Asia   
3   male  default           club               1               1    Asia   
4   male  default           club               1               1    Asia   

  country_code wyscout_id  
0          XAS       None  
1          XAS    

In [6]:
comps_clean.to_csv(os.path.join(output_folder, 'cleaned_competitions.csv'), index=False)

In [7]:
teams_df = pd.read_csv(os.path.join(input_folder, 'teams.csv'))
teams_clean = pd.merge(teams_df, areas_map, left_on='area_id', right_on='geo_id', how='left')

# Rename to match Unified Schema
teams_clean.rename(columns={
    'official_name': 'name',
    'image_data_url': 'logo_url'
}, inplace=True)
teams_clean['country'] = teams_clean['mapped_country']

# Purge technical and redundant columns but we keep 'type' (National vs Club)
teams_drop = ['name_x', 'name_y', 'area_id', 'geo_id', 'mapped_country', 'mapped_alpha3', 'created_at', 'updated_at']
teams_clean.drop(columns=[col for col in teams_drop if col in teams_clean.columns], inplace=True)
print(teams_clean.head())

   id                 name                                name          city  \
0   1            Pakruojis                        FC Pakruojis     Pakruojis   
1   2     Cúcuta Deportivo  Corporación Nuevo Cúcuta Deportivo        Cúcuta   
2   3         Saraburi TRU                     Saraburi TRU FC       Lopburi   
3   4  Achilleas Triandria                Achilleas Triandrias  Thessaloniki   
4   5                Løten                            Løten FK         Løten   

   type gender category                                           logo_url  \
0  club   male  default  https://cdn5.wyscout.com/photos/team/public/g3...   
1  club   male  default  https://cdn5.wyscout.com/photos/team/public/48...   
2  club   male  default  https://cdn5.wyscout.com/photos/team/public/nd...   
3  club   male  default  https://cdn5.wyscout.com/photos/team/public/nd...   
4  club   male  default  https://cdn5.wyscout.com/photos/team/public/nd...   

   parent_team_id  stadium  stadium_capacity    co

In [8]:
teams_clean.to_csv(os.path.join(output_folder, 'cleaned_teams.csv'), index=False)

In [9]:
players_df = pd.read_csv(os.path.join(input_folder, 'players.csv'))

# Combining first and last name safely
players_df['name'] = players_df['first_name'].fillna('') + ' ' + players_df['last_name'].fillna('')
players_df['name'] = players_df['name'].str.strip() # Remove extra spaces if someone only has one name

players_clean = pd.merge(players_df, areas_map, left_on='citizenship_area_id', right_on='geo_id', how='left')

# Rename to match Unified Schema
players_clean['nationality'] = players_clean['mapped_country']
players_clean.rename(columns={'role_name': 'position'}, inplace=True)

# Purge technical and redundant columns
players_drop = [
    'first_name', 'last_name', 'citizenship_area_id', 'second_citizenship_area_id', 
    'geo_id', 'mapped_country', 'mapped_alpha3', 'created_at', 'updated_at'
]
players_clean.drop(columns=[col for col in players_drop if col in players_clean.columns], inplace=True)
print(players_clean.head())

   id  short_name  birth_date  age_years  height_cm  weight_kg   foot  \
0   5  T. De Rijk  1992-03-31       33.0          0          0   left   
1   9  D. Liefden  1990-05-19       35.0          0          0  right   
2  10   C. N'Toko  1988-01-30       37.0        189         84  right   
3  12  K. Schutte  1992-08-02       33.0          0          0    NaN   
4  16   S. Köksal  1990-02-18       35.0        177         81  right   

     position role_code  status                    name  nationality  
0    Defender       DEF  active             Tim De Rijk  Netherlands  
1    Defender       DEF  active           Diego Liefden  Netherlands  
2    Defender       DEF  active  Chiró Mena Vuza-N'Toko     Congo DR  
3    Defender       DEF  active           Kevin Schutte  Netherlands  
4  Midfielder       MID  active           Serhat Köksal  Netherlands  


In [10]:
players_clean.to_csv(os.path.join(output_folder, 'cleaned_players.csv'), index=False)

# Table Matching

## Leagues

In [30]:
BASE_FILE = 'data/soccersolver-player-data/unified/unified_leagues.csv' 
TARGET_FILE = 'data/wyscout_data/cleaned/cleaned_competitions.csv' 
OUTPUT_BASE = 'data/unified_tables/leagues'
PATH_MATCHED = os.path.join(OUTPUT_BASE, 'matched')
PATH_PARTIAL = os.path.join(OUTPUT_BASE, 'partial match')
PATH_NO_MATCH = os.path.join(OUTPUT_BASE, 'no match')

# Create the directories
for path in [PATH_MATCHED, PATH_PARTIAL, PATH_NO_MATCH]:
    os.makedirs(path, exist_ok=True)


base_df = pd.read_csv(BASE_FILE)
target_df = pd.read_csv(TARGET_FILE)

# ID Standardisation
base_df.rename(columns={'league_id': 'soccersolver_id'}, inplace=True)

if 'wyscout_id' in target_df.columns:
    target_df.drop(columns=['wyscout_id'], inplace=True)
if 'id' in target_df.columns:
    target_df.rename(columns={'id': 'wyscout_id'}, inplace=True)


In [31]:
# Standardisation
def standardize_text(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    # Removes accents (e.g., 'é' -> 'e')
    text = ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')
    return text

base_df['match_name'] = base_df['name'].apply(standardize_text)
base_df['match_country'] = base_df['country'].apply(standardize_text)
target_df['match_name'] = target_df['name'].apply(standardize_text)
target_df['match_country'] = target_df['country'].apply(standardize_text)

# ONLY domestic leagues
if 'competition_type' in target_df.columns:
    target_df = target_df[target_df['competition_type'] == 'Domestic League'].copy()


In [32]:
# Matching Engine 
perfect_matches = []
partial_matches = []
handled_indices = set() # Tracks SoccerSolver rows safely

for t_idx, t_row in target_df.iterrows():
    best_score = 0
    best_base_idx = -1
    
    # Hard country filter
    candidates = base_df[base_df['match_country'] == t_row['match_country']]
    
    for b_idx, b_row in candidates.iterrows():
        score = fuzz.token_sort_ratio(t_row['match_name'], b_row['match_name'])
        if score > best_score:
            best_score = score
            best_base_idx = b_idx
            
    if best_base_idx != -1:
        base_row = base_df.loc[best_base_idx]
        if best_score >= 95:
            # Dynamically grab all columns from base, prefix them, and drop the helper columns
            base_dict = {f"soccersolver_{k}": v for k, v in base_row.items() if k not in ['match_name', 'match_country']}
            # Dynamically grab all columns from target, prefix them, and drop the helper columns
            target_dict = {f"wyscout_{k}": v for k, v in t_row.items() if k not in ['match_name', 'match_country']}
            
            # Fuse the two massive dictionaries together into one wide row
            combined_row = {**base_dict, **target_dict}
            combined_row['match_score'] = best_score
            
            perfect_matches.append(combined_row)
            handled_indices.add(t_idx) 
        
        # Pass 2: Partial Match
        elif best_score >= 75:
            base_dict = {f"soccersolver_{k}": v for k, v in base_row.items() if k not in ['match_name', 'match_country']}
            target_dict = {f"wyscout_{k}": v for k, v in t_row.items() if k not in ['match_name', 'match_country']}
            
            combined_row = {**base_dict, **target_dict}
            combined_row['match_score'] = best_score
            combined_row['APPROVED'] = '' # Blank column for manual review
            
            partial_matches.append(combined_row)
            handled_indices.add(t_idx)

In [33]:
# No Match
target_orphans = target_df.drop(index=list(handled_indices))
# using the new double-prefixed key, and check perfect and partial matches
matched_base_ids = [m['soccersolver_soccersolver_id'] for m in perfect_matches]
partial_base_ids = [m['soccersolver_soccersolver_id'] for m in partial_matches]

# Combining them so we know every single SoccerSolver row that has been accounted for
all_handled_base_ids = matched_base_ids + partial_base_ids

# Filter out the handled ones to leave only the true orphans
base_orphans = base_df[~base_df['soccersolver_id'].isin(all_handled_base_ids)]


In [34]:
pd.DataFrame(perfect_matches).to_csv(os.path.join(PATH_MATCHED, 'leagues_matched.csv'), index=False)    # Perfect Matches
pd.DataFrame(partial_matches).to_csv(os.path.join(PATH_PARTIAL, 'leagues_partial.csv'), index=False)    #partial matches

target_orphans.drop(columns=['match_name', 'match_country'], errors='ignore').to_csv(os.path.join(PATH_NO_MATCH, 'wyscout_leagues_unmatched.csv'), index=False)
base_orphans.drop(columns=['match_name', 'match_country'], errors='ignore').to_csv(os.path.join(PATH_NO_MATCH, 'soccersolver_leagues_unmatched.csv'), index=False)

print(" All data routed with 0 data loss.")
print(f"- Perfect Matches: {len(perfect_matches)}")
print(f"- Require Manual Review: {len(partial_matches)}")
print(f"- Unmatched SoccerSolver: {len(base_orphans)}")
print(f"- Unmatched Wyscout: {len(target_orphans)}")

 All data routed with 0 data loss.
- Perfect Matches: 30
- Require Manual Review: 45
- Unmatched SoccerSolver: 224
- Unmatched Wyscout: 1876


In [2]:
# Paths 
PATH_MATCHED = 'data/unified_tables/leagues/matched/leagues_matched.csv'
PATH_PARTIAL = 'data/unified_tables/leagues/partial match/leagues_partial.csv'
PATH_NO_MATCH_WYSCOUT = 'data/unified_tables/leagues/no match/wyscout_leagues_unmatched.csv'
PATH_NO_MATCH_SS = 'data/unified_tables/leagues/no match/soccersolver_leagues_unmatched.csv'

# Checking if there is actually anything to integrate
if not os.path.exists(PATH_PARTIAL):
    print("Error: leagues_partial.csv not found.")
    exit()

# Datasets
partial_df = pd.read_csv(PATH_PARTIAL)
matched_df = pd.read_csv(PATH_MATCHED)
unmatched_w_df = pd.read_csv(PATH_NO_MATCH_WYSCOUT)
unmatched_ss_df = pd.read_csv(PATH_NO_MATCH_SS)

# Clean and Filter
partial_df['APPROVED'] = pd.to_numeric(partial_df['APPROVED'], errors='coerce').fillna(0)       # Safely convert the APPROVED column to numbers. Blanks/NaNs become 0.

# Splitting the dataset
approved_df = partial_df[partial_df['APPROVED'] == 1].copy()
rejected_df = partial_df[partial_df['APPROVED'] != 1].copy()
print(f"Found {len(approved_df)} Approved Matches and {len(rejected_df)} Rejected Matches.")

# Processing Approved Rows
if not approved_df.empty:
    approved_df.drop(columns=['APPROVED'], inplace=True)        # Dropping the APPROVED column so it matches the schema of matched_df perfectly
    matched_df = pd.concat([matched_df, approved_df], ignore_index=True)        # Append to the bottom of the perfect matches
    print("Approved matches added to Matched folder.")

# Processing Rejected Rows
if not rejected_df.empty:
    ss_cols = [c for c in rejected_df.columns if c.startswith('soccersolver_')]     # Get only columns that start with 'soccersolver_'
    ss_rejected = rejected_df[ss_cols].copy()
    ss_rejected.rename(columns=lambda x: x.replace('soccersolver_', '', 1), inplace=True)       # Strip the prefix to return to original column names
    ss_rejected.drop_duplicates(subset=['soccersolver_id'], inplace=True)   # Drop duplicates in case a season was partially matched multiple times
    unmatched_ss_df = pd.concat([unmatched_ss_df, ss_rejected], ignore_index=True)  # Append to SoccerSolver orphans

    
    w_cols = [c for c in rejected_df.columns if c.startswith('wyscout_')]
    w_rejected = rejected_df[w_cols].copy()
    w_rejected.rename(columns=lambda x: x.replace('wyscout_', '', 1), inplace=True)  # Strip the prefix
    w_rejected.drop_duplicates(subset=['wyscout_id'], inplace=True)
    unmatched_w_df = pd.concat([unmatched_w_df, w_rejected], ignore_index=True) # Append to Wyscout orphans
    
    print("Rejected matches cleanly split and added to No Match folder.")



matched_df.to_csv(PATH_MATCHED, index=False)
unmatched_w_df.to_csv(PATH_NO_MATCH_WYSCOUT, index=False)
unmatched_ss_df.to_csv(PATH_NO_MATCH_SS, index=False)

#Renaming the partial file so you don't accidentally run this script twice
archive_path = PATH_PARTIAL.replace('.csv', '_PROCESSED.csv')
os.rename(PATH_PARTIAL, archive_path)

print("INTEGRATION COMPLETE")

Found 8 Approved Matches and 37 Rejected Matches.
Approved matches added to Matched folder.
Rejected matches cleanly split and added to No Match folder.
INTEGRATION COMPLETE


## Teams

In [19]:
BASE_FILE = 'data/soccersolver-player-data/unified/unified_teams.csv'  # SoccerSolver teams
TARGET_FILE = 'data/wyscout_data/cleaned/cleaned_teams.csv' # Wyscout teams

OUTPUT_BASE = 'data/unified_tables/teams'
PATH_MATCHED = os.path.join(OUTPUT_BASE, 'matched')
PATH_PARTIAL = os.path.join(OUTPUT_BASE, 'partial match')
PATH_NO_MATCH = os.path.join(OUTPUT_BASE, 'no match')

for path in [PATH_MATCHED, PATH_PARTIAL, PATH_NO_MATCH]:
    os.makedirs(path, exist_ok=True)

base_df = pd.read_csv(BASE_FILE)
target_df = pd.read_csv(TARGET_FILE)

# SoccerSolver ID Standardization
if 'team_id' in base_df.columns:
    base_df.rename(columns={'team_id': 'soccersolver_id'}, inplace=True)

print(f"Base rows before deduplication: {len(base_df)}")
if 'season' in base_df.columns:
    base_df = base_df.drop_duplicates(subset=['soccersolver_id', 'season'], keep='first')
print(f"Base rows after deduplication: {len(base_df)}")

# Wyscout ID 
if 'wyscout_id' in target_df.columns:
    target_df.drop(columns=['wyscout_id'], inplace=True)
if 'id' in target_df.columns:
    target_df.rename(columns={'id': 'wyscout_id'}, inplace=True)

# Filter: Clubs (Ignore National teams)
target_df = target_df[target_df['type'] == 'club'].copy()

# Standardization
def standardize_text(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    return ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')

if 'name.1' in target_df.columns:
    target_df['combined_name'] = target_df['name'].fillna('') + " " + target_df['name.1'].fillna('')
else:
    target_df['combined_name'] = target_df['name'].fillna('')


Base rows before deduplication: 9926
Base rows after deduplication: 9279


In [20]:
# Standardize Base (SoccerSolver)
base_df['match_name'] = base_df['name'].apply(standardize_text)
base_df['match_country'] = base_df['country'].apply(standardize_text)
base_df['match_stadium'] = base_df['stadium_name'].apply(standardize_text)

# Standardize Target (Wyscout)
target_df['match_name'] = target_df['combined_name'].apply(standardize_text)
target_df['match_country'] = target_df['country'].apply(standardize_text)
target_df['match_stadium'] = target_df['stadium'].apply(standardize_text)

# The Semantic Geographic Dictionary
country_mapping = {
    "united kingdom": "england",
    "uk": "england",
    "usa": "united states",
    "holland": "netherlands",
    "republic of ireland": "ireland",
    "korea republic": "south korea",
    "pr china": "china",
    "peoples republic of china": "china",
    "cote divoire": "ivory coast",  # Standardized text strips the apostrophe
    "bosnia-herzegovina": "bosnia and herzegovina",
    "bosnia": "bosnia and herzegovina",
    "turkiye": "turkey",
    "czechia": "czech republic",
    "macedonia": "north macedonia",
    "cape verde": "cabo verde"
}

base_df['match_country'] = base_df['match_country'].replace(country_mapping)
target_df['match_country'] = target_df['match_country'].replace(country_mapping)

# Columns to drop when fusing rows to keep the CSV clean
helper_cols = ['match_name', 'match_country', 'match_stadium', 'combined_name']
if 'name.1' in target_df.columns:
    helper_cols.append('name.1')

In [21]:
# Checking for absolute duplicates (where EVERY column is identical)
exact_duplicates = base_df.duplicated(subset=['soccersolver_id', 'season'], keep=False)
print(f"Exact identical seasonal rows found: {exact_duplicates.sum()}")

# Checking the variance of numerical columns across seasons
variance_check = base_df.groupby('soccersolver_id')['total_market_value'].var()     # We group by the team ID and measure how much their Market Value changed over the years

# If variance is 0, the market value never changed across all 7 seasons
static_teams = variance_check[variance_check == 0]
print(f"Teams with ZERO change in market value across seasons: {len(static_teams)}")

age_variance = base_df.groupby('soccersolver_id')['average_age'].var()
print(f"Average Age variance across dataset: {age_variance.mean():.2f}")

Exact identical seasonal rows found: 0
Teams with ZERO change in market value across seasons: 16
Average Age variance across dataset: 0.90


In [25]:
#matching
perfect_matches = []
partial_matches = []
handled_target_indices = set()
handled_base_indices = set()

#  current time for the "In-Data" tracking column
processing_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print(f" Processing {len(target_df)} Wyscout teams...")

# live progress bar 
for t_idx, t_row in tqdm(target_df.iterrows(), total=len(target_df), desc="Matching Teams"):
    
    # EXACT match on country (optimized to avoid the hang)
    candidates = base_df[base_df['match_country'] == t_row['match_country']]
    
    for b_idx, b_row in candidates.iterrows():
        name_score = fuzz.token_set_ratio(t_row['match_name'], b_row['match_name'])
        
        t_stad = str(t_row['match_stadium']) if pd.notna(t_row['match_stadium']) else ""
        b_stad = str(b_row['match_stadium']) if pd.notna(b_row['match_stadium']) else ""
        stadium_score = fuzz.token_set_ratio(t_stad, b_stad) if t_stad and b_stad else 0
        
        is_perfect = (name_score >= 95) or (name_score >= 85 and stadium_score >= 90)
        
        if is_perfect or name_score >= 75:
            base_dict = {f"soccersolver_{k}": v for k, v in b_row.items() if k not in helper_cols}
            target_dict = {f"wyscout_{k}": v for k, v in t_row.items() if k not in helper_cols}
            combined_row = {**base_dict, **target_dict}
            
            # Tracking Data
            combined_row['match_score'] = name_score
            combined_row['processed_at'] = processing_time # Live tracking in the data
            
            if is_perfect:
                perfect_matches.append(combined_row)
                handled_target_indices.add(t_idx)
                handled_base_indices.add(b_idx)
            else:
                combined_row['APPROVED'] = '' 
                partial_matches.append(combined_row)
                handled_target_indices.add(t_idx)
                handled_base_indices.add(b_idx)

 Processing 35086 Wyscout teams...


Matching Teams:   0%|          | 0/35086 [00:00<?, ?it/s]

Matching Teams: 100%|██████████| 35086/35086 [05:09<00:00, 113.23it/s]


In [26]:
pd.DataFrame(perfect_matches).to_csv(os.path.join(PATH_MATCHED, 'teams_matched.csv'), index=False)
pd.DataFrame(partial_matches).to_csv(os.path.join(PATH_PARTIAL, 'teams_partial.csv'), index=False)

target_orphans.drop(columns=helper_cols, errors='ignore').to_csv(os.path.join(PATH_NO_MATCH, 'wyscout_teams_unmatched.csv'), index=False)
base_orphans.drop(columns=helper_cols, errors='ignore').to_csv(os.path.join(PATH_NO_MATCH, 'soccersolver_teams_unmatched.csv'), index=False)

print("Teams data successfully mapped and routed.")
print(f"- Perfect Matches: {len(perfect_matches)}")
print(f"- Require Manual Review: {len(partial_matches)}")
print(f"- Unmatched SoccerSolver: {len(base_orphans)}")
print(f"- Unmatched Wyscout: {len(target_orphans)}")

Teams data successfully mapped and routed.
- Perfect Matches: 28911
- Require Manual Review: 23128
- Unmatched SoccerSolver: 8192
- Unmatched Wyscout: 34853


## Players

In [7]:
BASE_FILE = 'data/soccersolver-player-data/unified/merged_players.csv' 
TARGET_FILE = 'data/wyscout_data/cleaned/cleaned_players.csv'
OUTPUT_BASE = 'data/unified_tables/players'

PATH_MATCHED = os.path.join(OUTPUT_BASE, 'matched')
PATH_PARTIAL = os.path.join(OUTPUT_BASE, 'partial match')
PATH_NO_MATCH = os.path.join(OUTPUT_BASE, 'no match')

for path in [PATH_MATCHED, PATH_PARTIAL, PATH_NO_MATCH]:
    os.makedirs(path, exist_ok=True)

# MAPPING 
POS_MAP = {'GK': 'GK', 'DEF': 'DEF', 'MID': 'MID', 'ATT': 'FWD'}
FOOT_MAP = {'right': 'right', 'left': 'left', 'both': 'both'}

def clean_text(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    return "".join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')

base_df = pd.read_csv(BASE_FILE)
target_df = pd.read_csv(TARGET_FILE)

base_df.rename(columns={'player_id': 'soccersolver_id'}, inplace=True, errors='ignore')
target_df.rename(columns={'id': 'wyscout_id'}, inplace=True, errors='ignore')

base_df['birth_date'] = pd.to_datetime(base_df['birth_date'], dayfirst=True, errors='coerce')
target_df['birth_date'] = pd.to_datetime(target_df['birth_date'], errors='coerce')


In [8]:
# DEDUPLICATION
print(f" DEDUPLICATING PROFILES")
base_unique = base_df.drop_duplicates(subset=['soccersolver_id']).copy()
print(f"[+] Squashed into {len(base_unique)} unique SoccerSolver profiles.")

# THE BIRTHDAY JOIN (MEMORY-SAFE ANCHOR)
print(f"PRIMARY ENGINE: BIRTHDATE ANCHOR")

# Isolate valid dates
base_valid = base_unique.dropna(subset=['birth_date']).copy()
target_valid = target_df.dropna(subset=['birth_date']).copy()

# Trim columns to prevent memory overflow during merge
base_cols = ['soccersolver_id', 'birth_date', 'name', 'nationality', 'height', 'position', 'preferred_foot']
target_cols = ['wyscout_id', 'birth_date', 'name', 'short_name', 'nationality', 'height_cm', 'role_code', 'foot']

base_subset = base_valid[base_cols].copy()
target_subset = target_valid[target_cols].copy()

# Execute primary valid join
merged = pd.merge(base_subset, target_subset, on='birth_date', suffixes=('_ss', '_wy'))
total_to_process = len(merged)
print(f"[!] Primary Engine found {total_to_process} potential matches to score.")

# --- 5B. THE FALLBACK ENGINE (RESCUING BLANKS) ---
print(f"\n{'='*40}\n[4/7] FALLBACK ENGINE: MISSING DATES\n{'='*40}")

# Isolate missing dates
base_missing = base_unique[base_unique['birth_date'].isna()].copy()
target_missing = target_df[target_df['birth_date'].isna()].copy()

print(f"[*] Rescuing from {len(base_missing)} SS blanks and {len(target_missing)} WY blanks...")

# Clean text for exact deterministic matching
base_missing['c_name'] = base_missing['name'].apply(clean_text)
base_missing['c_nat'] = base_missing['nationality'].apply(clean_text)

target_missing['c_name_full'] = target_missing['name'].apply(clean_text)
target_missing['c_name_short'] = target_missing['short_name'].apply(clean_text)
target_missing['c_nat'] = target_missing['nationality'].apply(clean_text)

# Exact Join 1: Full Name + Nationality
fallback_1 = pd.merge(
    base_missing[['soccersolver_id', 'c_name', 'c_nat']], 
    target_missing[['wyscout_id', 'c_name_full', 'c_nat']], 
    left_on=['c_name', 'c_nat'], right_on=['c_name_full', 'c_nat']
)

# Exact Join 2: Short Name + Nationality (Excluding already matched)
matched_fallback_ids = fallback_1['soccersolver_id'].unique()
base_missing_rem = base_missing[~base_missing['soccersolver_id'].isin(matched_fallback_ids)]

fallback_2 = pd.merge(
    base_missing_rem[['soccersolver_id', 'c_name', 'c_nat']], 
    target_missing[['wyscout_id', 'c_name_short', 'c_nat']], 
    left_on=['c_name', 'c_nat'], right_on=['c_name_short', 'c_nat']
)

fallback_merged = pd.concat([fallback_1, fallback_2], ignore_index=True)
print(f"[+] Fallback Engine successfully rescued {len(fallback_merged)} players without birthdays.")

# Initialize results list with the Fallback rescues
results = []
for _, row in fallback_merged.iterrows():
    results.append({
        'soccersolver_id': row['soccersolver_id'],
        'wyscout_id': row['wyscout_id'],
        'confidence_score': 75,   # Assigned to Partial for manual safety check
        'match_status': 'partial'
    })

#  WEIGHTED SCORING ENGINE 
print(f"SCORING ENGINE (LIVE TRACKER)")
start_time = pd.Timestamp.now()

# Itertuples bypasses the dictionary memory trap
for i, row in enumerate(merged.itertuples(index=False)):
    
    # --- Live Heartbeat Log ---
    if i % 1000 == 0 and i > 0:
        elapsed = (pd.Timestamp.now() - start_time).total_seconds()
        print(f"--- LIVE UPDATE: Processed {i}/{total_to_process} candidates | Time Elapsed: {elapsed:.1f}s ---")
        sys.stdout.flush()

    # 1. Name Score (60)
    name_ss = clean_text(row.name_ss)
    name_wy_f = clean_text(row.name_wy)
    name_wy_s = clean_text(row.short_name)
    score_name = (max(fuzz.token_sort_ratio(name_ss, name_wy_f), 
                      fuzz.token_sort_ratio(name_ss, name_wy_s)) / 100) * 60

    # 2. Nationality (20)
    score_nat = 20 if clean_text(row.nationality_ss) == clean_text(row.nationality_wy) else 0

    # 3. Height (10)
    try:
        diff = abs(float(row.height) - float(row.height_cm))
        score_height = 10 if diff <= 2 else (5 if diff <= 4 else 0)
    except: score_height = 0

    # 4. Position (5)
    try:
        score_pos = 5 if POS_MAP.get(row.position, '') == POS_MAP.get(row.role_code, '') else 0
    except: score_pos = 0

    # 5. Foot (5)
    try:
        foot_ss = FOOT_MAP.get(str(row.preferred_foot).lower(), '')
        foot_wy = FOOT_MAP.get(str(row.foot).lower(), '')
        score_foot = 5 if foot_ss == foot_wy and foot_ss != '' else 0
    except: score_foot = 0

    total_confidence = score_name + score_nat + score_height + score_pos + score_foot
    
    status = 'no_match'
    if total_confidence >= 80: status = 'perfect'
    elif total_confidence >= 60: status = 'partial'

    results.append({
        'soccersolver_id': row.soccersolver_id,
        'wyscout_id': row.wyscout_id,
        'confidence_score': total_confidence,
        'match_status': status
    })

match_results_df = pd.DataFrame(results).sort_values('confidence_score', ascending=False).drop_duplicates('soccersolver_id')

final_df = pd.merge(base_df, match_results_df, on='soccersolver_id', how='left')

# Drop Wyscout helper columns before merging to avoid duplicate "_x / _y" clutter
target_cols_to_keep = [c for c in target_df.columns if c not in ['birth_date', 'nationality', 'name', 'short_name']]
final_df = pd.merge(final_df, target_df[target_cols_to_keep], on='wyscout_id', how='left', suffixes=('_ss', '_wy'))

helper_cols = ['confidence_score', 'match_status']

# 1. Perfect
perfect = final_df[final_df['match_status'] == 'perfect'].copy()
perfect.drop(columns=helper_cols, errors='ignore').to_csv(os.path.join(PATH_MATCHED, 'matches_players.csv'), index=False)
print(f" Perfect Matches: {len(perfect)}")

# 2. Partial (With APPROVED column)
partial = final_df[final_df['match_status'] == 'partial'].copy()
partial['APPROVED'] = ""
partial.drop(columns=helper_cols, errors='ignore').to_csv(os.path.join(PATH_PARTIAL, 'partial_match.csv'), index=False)
print(f" Partial Matches (Manual Review): {len(partial)}")

# 3. Unmatched Orphans (SoccerSolver)
wy_specific_cols = [c for c in target_cols_to_keep if c != 'wyscout_id']
ss_orphans = final_df[final_df['match_status'].isna()].copy()
ss_orphans.drop(columns=helper_cols + wy_specific_cols, errors='ignore').to_csv(os.path.join(PATH_NO_MATCH, 'soccersolver_unmatched.csv'), index=False)
print(f" Unmatched SoccerSolver: {len(ss_orphans)}")

# 4. Unmatched Orphans (Wyscout)
matched_wy_ids = match_results_df['wyscout_id'].dropna().unique()
wy_orphans = target_df[~target_df['wyscout_id'].isin(matched_wy_ids)].copy()
wy_orphans.to_csv(os.path.join(PATH_NO_MATCH, 'wyscout_unmatched.csv'), index=False)
print(f"Unmatched Wyscout: {len(wy_orphans)}")


 DEDUPLICATING PROFILES
[+] Squashed into 114158 unique SoccerSolver profiles.
PRIMARY ENGINE: BIRTHDATE ANCHOR
[!] Primary Engine found 10120152 potential matches to score.

[4/7] FALLBACK ENGINE: MISSING DATES
[*] Rescuing from 1922 SS blanks and 90299 WY blanks...
[+] Fallback Engine successfully rescued 216 players without birthdays.
SCORING ENGINE (LIVE TRACKER)
--- LIVE UPDATE: Processed 1000/10120152 candidates | Time Elapsed: 0.1s ---
--- LIVE UPDATE: Processed 2000/10120152 candidates | Time Elapsed: 0.1s ---
--- LIVE UPDATE: Processed 3000/10120152 candidates | Time Elapsed: 0.1s ---
--- LIVE UPDATE: Processed 4000/10120152 candidates | Time Elapsed: 0.2s ---
--- LIVE UPDATE: Processed 5000/10120152 candidates | Time Elapsed: 0.2s ---
--- LIVE UPDATE: Processed 6000/10120152 candidates | Time Elapsed: 0.3s ---
--- LIVE UPDATE: Processed 7000/10120152 candidates | Time Elapsed: 0.3s ---
--- LIVE UPDATE: Processed 8000/10120152 candidates | Time Elapsed: 0.4s ---
--- LIVE UPDAT